# LLM Fine-Tuning: Customer Support Agent

Fine-tunes **Llama 3.2 3B Instruct** on 27K real customer support conversations using LoRA.

**Runtime**: T4 GPU (free Colab tier)  
**Time**: ~25 mins for 200 steps | ~2 hrs for full training  
**Dataset**: [Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset)

## Before running:
1. Runtime → Change runtime type → T4 GPU
2. Set your secrets: W&B key, HF token (left sidebar 🔑)


In [ ]:
# ── cell 1: check GPU ──────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_available(), 'Switch to T4 GPU: Runtime → Change runtime type'

In [ ]:
# ── cell 2: install dependencies ───────────────────────────────────────
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install wandb datasets evaluate rouge-score bert-score rich python-dotenv

In [ ]:
# ── cell 3: auth ───────────────────────────────────────────────────────
import os
from google.colab import userdata
import wandb

# set your keys in Colab Secrets (left sidebar 🔑)
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

wandb.login()
print('Auth complete')

In [ ]:
# ── cell 4: load model with Unsloth ───────────────────────────────────
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LENGTH = 2048
MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'
LORA_RANK = 16  # change to 64 for second experiment

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template='llama-3')
print(f'Model loaded: {MODEL_NAME}')

In [ ]:
# ── cell 5: apply LoRA ─────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} params ({trainable/total*100:.2f}%)')

In [ ]:
# ── cell 6: download + clean dataset ──────────────────────────────────
import pandas as pd
import re
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split

SYSTEM_PROMPT = (
    'You are a helpful, professional customer support agent. '
    'Respond clearly and empathetically to customer inquiries. '
    'Be concise, accurate, and solution-focused.'
)

# download
ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
df = ds.to_pandas()
print(f'Raw: {len(df):,} rows | intents: {df["intent"].nunique()} | categories: {df["category"].nunique()}')

# clean
df = df.dropna(subset=['instruction', 'response'])
df = df[df['response'].str.len() >= 20]
df = df[df['instruction'].str.len() >= 5]
df['instruction'] = df['instruction'].str.strip().str.replace(r'\s+', ' ', regex=True)
df['response'] = df['response'].str.strip().str.replace(r'\s+', ' ', regex=True)
df = df.drop_duplicates(subset=['instruction', 'response']).reset_index(drop=True)
print(f'After cleaning: {len(df):,} rows')

# split (stratified)
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['intent'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['intent'], random_state=42)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

# format for instruct tuning
def make_messages(row):
    return {'messages': [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': row['instruction']},
        {'role': 'assistant', 'content': row['response']},
    ]}

train_ds = Dataset.from_list([make_messages(r) for r in train_df.to_dict('records')])
val_ds = Dataset.from_list([make_messages(r) for r in val_df.to_dict('records')])

def apply_template(examples):
    texts = tokenizer.apply_chat_template(examples['messages'], tokenize=False, add_generation_prompt=False)
    return {'text': texts}

train_ds = train_ds.map(apply_template, batched=True)
val_ds = val_ds.map(apply_template, batched=True)
print('Dataset ready.')

In [ ]:
# ── cell 7: quick data sanity check ───────────────────────────────────
print('Sample training example:')
print('-' * 60)
print(train_ds[0]['text'][:500])
print('-' * 60)

In [ ]:
# ── cell 8: configure training ─────────────────────────────────────────
import wandb
from trl import SFTTrainer
from transformers import TrainingArguments

EXPERIMENT_NAME = f'customer-support-lora-r{LORA_RANK}'
MAX_STEPS = 200  # set to 500+ for full training

wandb.init(
    project='llm-finetuning-customer-support',
    name=EXPERIMENT_NAME,
    config={'model': MODEL_NAME, 'lora_rank': LORA_RANK, 'max_steps': MAX_STEPS},
)

training_args = TrainingArguments(
    output_dir=f'./outputs/{EXPERIMENT_NAME}',
    max_steps=MAX_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,
    seed=42,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    report_to='wandb',
    run_name=EXPERIMENT_NAME,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)
print('Trainer configured. Starting training...')

In [ ]:
# ── cell 9: TRAIN ──────────────────────────────────────────────────────
import time
start = time.time()
trainer.train()
elapsed = (time.time() - start) / 60
print(f'Training done in {elapsed:.1f} minutes')
wandb.log({'train/total_minutes': elapsed})

In [ ]:
# ── cell 10: quick inference test ─────────────────────────────────────
FastLanguageModel.for_inference(model)

def chat(instruction):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': instruction},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

test_questions = [
    'I was charged twice for my order. What should I do?',
    'How do I return a product I bought last week?',
    'My account is locked. Help!',
]

for q in test_questions:
    print(f'Q: {q}')
    print(f'A: {chat(q)}')
    print('-' * 60)

In [ ]:
# ── cell 11: save + push to HuggingFace ───────────────────────────────
import os

# save locally
model.save_pretrained(f'outputs/{EXPERIMENT_NAME}')
tokenizer.save_pretrained(f'outputs/{EXPERIMENT_NAME}')

# push to Hub (so deployment app can use it)
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    repo_id = f'vamsiyvk/{EXPERIMENT_NAME}'
    model.push_to_hub(repo_id, token=hf_token)
    tokenizer.push_to_hub(repo_id, token=hf_token)
    print(f'Pushed to: https://huggingface.co/{repo_id}')
else:
    print('HF_TOKEN not set — skipping Hub push')

wandb.finish()
print('Done!')

In [ ]:
# ── cell 12: save test set for evaluation ─────────────────────────────
test_df.to_parquet('test_set.parquet', index=False)
print(f'Test set saved: {len(test_df)} rows')
print('Download test_set.parquet and run evaluation/automated_eval.py locally')